<a href="https://colab.research.google.com/github/itsahab01/Commercial-Law-Semantic-Search/blob/main/Commercial_Law.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="
background:#0B5D3B;
color:white;
padding:22px;
border-radius:14px;
font-family:Arial;
">

<div style="
background:#f5f7f6;
color:#222;
padding:18px;
border-radius:10px;
">

<h2 style="margin:0; color:#0B5D3B;">Commercial Law Semantic Search</h2>

<p style="margin:8px 0 0 0; font-size:14px;">
RAG-based system for retrieving and answering queries from Saudi commercial law documents.
</p>

<p style="margin:10px 0 0 0; font-size:13px; opacity:0.85;">
This project uses semantic embeddings and FAISS indexing to perform accurate legal search over Arabic text.
</p>

<hr style="margin:15px 0; border-color:#00000022;">

<p style="margin:0; font-size:13px;">
<strong>Tech Stack:</strong> RAG • Sentence Transformers • FAISS • Arabic NLP
</p>

</div>

</div>


#  Project Overview

This project implements a simple Retrieval-Augmented Generation (RAG) system
to search within the Saudi Commercial Register Law document.

Instead of relying on keyword matching, the system uses semantic search
to understand the meaning of queries and retrieve the most relevant legal text.

---

##  Objectives

- Convert legal text into embeddings (vector representation)
- Store embeddings using FAISS for fast retrieval
- Perform semantic search using cosine similarity
- Return the most relevant legal evidence for a given query

---

##  Why this matters

Legal documents are long and complex. Traditional search fails when the query
is phrased differently. This system solves that using AI-powered semantic understanding.

# Step 0: Install and Load the embedding model

In [ ]:
!pip -q install -U "sentence-transformers>=3.0" "faiss-cpu>=1.8"
print("Setup complete")

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID)

print("Loaded:", MODEL_ID)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

# Step 1: Load the Legal Document

We load the Saudi Commercial Register Law.
This will be the knowledge base for our system.

> ⚠️ **Important Notice**  
> Please make sure to upload the dataset file (`data.txt`) before running the notebook

In [ ]:
with open("/content/data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:300])

#  Step 2: Text Chunking

Legal documents are long, and embedding models have token limits.

We split the document into smaller overlapping chunks:
- Each chunk contains a fixed number of words
- Overlap ensures context is not lost between chunks

This improves retrieval accuracy.

In [ ]:
def chunk_text(text, size=80, overlap=30):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + size]))
        i += size - overlap
    return chunks

chunks = chunk_text(text)
print("Number of chunks:", len(chunks))
print(chunks[0])

#  Step 3: Generate Embeddings + cosine similarity

Each chunk is converted into a numerical vector using a pre-trained model.

These embeddings capture the semantic meaning of the text,
so similar meanings are closer in vector space.

In [ ]:
import numpy as np

embeddings = model.encode(chunks, normalize_embeddings=True)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(cosine(embeddings[0], embeddings[1]))

#  Step 4: Build FAISS Index

We store all embeddings in a FAISS index.

FAISS allows fast similarity search using:
- Inner Product (dot product)
- Equivalent to cosine similarity after normalization

This enables efficient retrieval of relevant text.

In [ ]:
import faiss

embeddings = np.array(embeddings).astype("float32")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Vectors in index:", index.ntotal)

#  Step 5: Semantic search

We convert the user question into an embedding
using the same model.

Then we search for the top-k most similar chunks
from the FAISS index.

In [ ]:
import textwrap #لتعديل تنسيق النص

def semantic_search(query, k=3):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(float(scores[0][r]), chunks[ids[0][r]]) for r in range(k)]

query = "متى يعلق قيد التاجر في السجل التجاري؟"
print("\n السؤال:\n", query, "\n")

for score, chunk in semantic_search(query, k=3):
    lines = textwrap.wrap(chunk, width=100)
    print("\n", f"{score:.3f} | {lines[0][::1]}")

    if len(lines) > 1:
        print("      ", lines[1][::1])

# Step 6: Keyword vs Semantic

In [ ]:
def keyword_search(query, docs, k=3):
    q_words = set(query.split())
    scored = [(len(q_words & set(d.split())), d) for d in docs]
    return sorted(scored, reverse=True)[:k]

query = "هل التاجر يتحمل مسؤولية لو المعلومات غلط؟"
print(keyword_search(query, chunks, k=2))
print(semantic_search(query, k=2))

#  Recap

- We loaded a real legal document
- Split it into chunks
- Converted text into embeddings
- Stored vectors using FAISS
- Performed semantic search

---

##  Key Insight

Semantic search works even when:
- The query uses different wording
- The answer is phrased differently in the document

This is why RAG is powerful.